# Belajar Data Mining Bertahap: Light -> Medium -> High -> Extra High

Notebook ini adalah jalur belajar **praktis dan berurutan**. Jalankan sel dari atas ke bawah. Setiap tingkat memakai dataset bawaan `scikit-learn`, sehingga dapat dijalankan tanpa mengunduh data dari internet.

| Tingkat | Fokus | Hasil yang diharapkan |
|---|---|---|
| Light | mengenal data dan EDA | memahami bentuk, kualitas, dan pola awal data |
| Medium | model klasifikasi dan evaluasi | membuat baseline yang benar tanpa data leakage |
| High | tuning, ketidakseimbangan kelas, interpretasi | memilih model secara lebih andal |
| Extra High | alur proyek tabular yang dapat direproduksi | siap mengerjakan dataset CSV sendiri |

> **Catatan penting:** Data mining bukan sekadar mencari akurasi tertinggi. Data harus dipisahkan sebelum preprocessing/modeling agar evaluasi tidak bias (*data leakage*).

## 0. Persiapan

Jika impor berikut gagal, instal paket di terminal dengan: `python3 -m pip install pandas numpy matplotlib scikit-learn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, load_breast_cancer, make_classification
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, classification_report, ConfusionMatrixDisplay,
    balanced_accuracy_score, average_precision_score,
    silhouette_score, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option('display.max_columns', 20)

# Level Light  -  Memahami Data

## Tujuan

1. Memahami masalah dan target.
2. Memeriksa tipe data, nilai hilang, duplikat, dan distribusi.
3. Membuat visualisasi sederhana.

Gunakan kerangka **CRISP-DM**: pemahaman bisnis -> pemahaman data -> persiapan data -> pemodelan -> evaluasi -> penerapan. Jangan membuat model sebelum memahami arti setiap kolom.

In [ ]:
# Dataset Iris: prediksi spesies bunga dari ukuran sepal dan petal
iris = load_iris(as_frame=True)
df_iris = iris.frame.copy()
df_iris['species'] = df_iris['target'].map(dict(enumerate(iris.target_names)))

print('Ukuran data:', df_iris.shape)
display(df_iris.head())
display(df_iris.dtypes)
display(df_iris.describe().T)

In [ ]:
# Pemeriksaan kualitas data paling dasar
quality_iris = pd.DataFrame({
    'missing': df_iris.isna().sum(),
    'unique': df_iris.nunique(),
    'dtype': df_iris.dtypes.astype(str)
})
display(quality_iris)
print('Jumlah baris duplikat:', df_iris.duplicated().sum())
display(df_iris['species'].value_counts())

In [ ]:
# EDA: lihat pemisahan dua fitur terhadap spesies
fig, ax = plt.subplots(figsize=(7, 5))
for species, group in df_iris.groupby('species'):
    ax.scatter(group['petal length (cm)'], group['petal width (cm)'], label=species, alpha=0.75)
ax.set(xlabel='Panjang petal (cm)', ylabel='Lebar petal (cm)', title='Iris: pola awal per spesies')
ax.legend(title='Spesies')
plt.show()

### Cek pemahaman Light

- Target pada contoh ini adalah `target` (atau `species` yang merupakan versi berlabel dari target). **Jangan masukkan `species` sebagai fitur** karena ia membocorkan jawaban.
- Dari grafik, spesies mana yang paling mudah dipisahkan? Apakah ada dua kelompok yang tumpang tindih?
- Untuk CSV sendiri, mulai dengan `df = pd.read_csv('nama_file.csv')`, lalu ulangi tiga pemeriksaan kualitas di atas.

# Level Medium  -  Klasifikasi dan Evaluasi yang Benar

Kita membuat model K-Nearest Neighbors (KNN). KNN bergantung pada jarak, jadi fiturnya perlu distandardisasi. `Pipeline` memastikan scaler hanya dipelajari dari data latih pada setiap proses validasi.

In [ ]:
# Pisahkan fitur (X) dan target (y); target tidak boleh ada di dalam X
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier(n_neighbors=5))
])
knn_pipeline.fit(X_train, y_train)
y_pred = knn_pipeline.predict(X_test)

print(f'Akurasi data uji: {accuracy_score(y_test, y_pred):.3f}')
print(classification_report(y_test, y_pred, target_names=iris.target_names, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=iris.target_names, cmap='Blues')
plt.title('Confusion matrix: KNN pada Iris')
plt.show()

In [ ]:
# Cross-validation: estimasi performa yang lebih stabil daripada satu kali split
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(knn_pipeline, X, y, cv=cv, scoring='accuracy')
print('Skor tiap fold:', np.round(scores, 3))
print(f'Rata-rata +/- standar deviasi: {scores.mean():.3f} +/- {scores.std():.3f}')

### Cara membaca metrik

- **Accuracy**: proporsi prediksi benar; cukup baik jika kelas relatif seimbang.
- **Precision**: dari semua prediksi positif, berapa yang benar.
- **Recall**: dari semua positif yang sebenarnya, berapa yang ditemukan.
- **F1-score**: keseimbangan precision dan recall.

Untuk masalah fraud/penyakit dengan kelas positif jarang, jangan hanya memakai accuracy; gunakan minimal recall, precision, F1, atau PR-AUC.

# Level High  -  Tuning, Kelas Tidak Seimbang, dan Interpretasi

Dataset Breast Cancer dipakai sebagai contoh klasifikasi biner. Kita akan: (1) menyetel hiperparameter pada data latih dengan cross-validation, (2) mengevaluasi sekali pada data uji yang tidak pernah dilihat, dan (3) melihat pentingnya fitur dengan permutation importance.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
X_cancer = cancer.data
y_cancer = cancer.target

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cancer, y_cancer, test_size=0.25, stratify=y_cancer, random_state=RANDOM_STATE
)
print('Distribusi target (0=malignant, 1=benign):')
display(y_cancer.value_counts(normalize=True).rename('proporsi'))

In [ ]:
# GridSearchCV harus menerima pipeline, bukan data yang sudah di-scale seluruhnya
logreg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE))
])

grid = GridSearchCV(
    estimator=logreg_pipeline,
    param_grid={'model__C': [0.01, 0.1, 1, 10]},
    scoring='balanced_accuracy',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1
)
grid.fit(X_train_c, y_train_c)

best_model = grid.best_estimator_
test_pred = best_model.predict(X_test_c)
test_proba = best_model.predict_proba(X_test_c)[:, 1]
print('Parameter terbaik:', grid.best_params_)
print(f'CV balanced accuracy terbaik: {grid.best_score_:.3f}')
print(f'Test balanced accuracy: {balanced_accuracy_score(y_test_c, test_pred):.3f}')
print(f'Test average precision (PR-AUC): {average_precision_score(y_test_c, test_proba):.3f}')

In [ ]:
# Kurva ROC dan Precision-Recall: lihat trade-off dari berbagai threshold
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(y_test_c, test_proba, ax=axes[0])
PrecisionRecallDisplay.from_predictions(y_test_c, test_proba, ax=axes[1])
axes[0].set_title('ROC curve')
axes[1].set_title('Precision-Recall curve')
plt.tight_layout()
plt.show()

In [ ]:
# Pentingnya fitur: penurunan skor ketika satu kolom diacak
perm = permutation_importance(
    best_model, X_test_c, y_test_c, scoring='balanced_accuracy',
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)
importance = pd.Series(perm.importances_mean, index=X_cancer.columns).sort_values(ascending=False).head(10)
display(importance.rename('mean_importance').to_frame())
importance.sort_values().plot.barh(figsize=(8, 5), title='10 fitur teratas menurut permutation importance')
plt.xlabel('Penurunan balanced accuracy saat fitur diacak')
plt.show()

# Penting: feature importance menunjukkan asosiasi/pengaruh pada model, bukan sebab-akibat.

## Clustering singkat (unsupervised)

Pada clustering tidak ada target. Nilai silhouette membantu membandingkan jumlah cluster, tetapi keputusan akhir juga harus mempertimbangkan konteks bisnis dan interpretasi cluster.

In [ ]:
# Standardisasi diperlukan karena K-Means memakai jarak
X_cluster = StandardScaler().fit_transform(iris.data)
silhouettes = {}
for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE).fit_predict(X_cluster)
    silhouettes[k] = silhouette_score(X_cluster, labels)

pd.Series(silhouettes, name='silhouette_score').plot(kind='bar', ylim=(0, 1), title='Perbandingan jumlah cluster')
plt.ylabel('Silhouette score (lebih tinggi umumnya lebih baik)')
plt.show()

# Level Extra High  -  Template Proyek Tabular yang Reproducible

Contoh berikut mensimulasikan data pelanggan dengan fitur numerik, kategori, dan nilai hilang. Pola ini dapat dipakai untuk CSV Anda sendiri. Kunci desainnya: `ColumnTransformer` menangani jenis kolom berbeda; seluruh preprocessing berada di dalam `Pipeline`; data uji disimpan sampai evaluasi akhir.

In [ ]:
# Data contoh yang deterministik; pada proyek nyata ganti bagian ini dengan pd.read_csv(...)
X_numeric, y_project = make_classification(
    n_samples=600, n_features=5, n_informative=3, n_redundant=1,
    weights=[0.65, 0.35], flip_y=0.03, random_state=RANDOM_STATE
)
rng = np.random.default_rng(RANDOM_STATE)
df_project = pd.DataFrame(X_numeric, columns=['tenure_score', 'usage_score', 'support_score', 'payment_score', 'engagement_score'])
df_project['plan'] = rng.choice(['basic', 'pro', 'enterprise'], size=len(df_project), p=[0.50, 0.35, 0.15])
df_project['region'] = rng.choice(['barat', 'tengah', 'timur'], size=len(df_project))
df_project.loc[rng.choice(df_project.index, size=25, replace=False), 'support_score'] = np.nan
df_project['churn'] = y_project
display(df_project.head())
print('Missing per kolom:')
display(df_project.isna().sum())

In [ ]:
# 1) Definisikan target dan fitur. Hindari kolom ID, timestamp pasca-kejadian, atau target terselubung.
target_col = 'churn'
X_project = df_project.drop(columns=target_col)
y_project = df_project[target_col]

numeric_features = X_project.select_dtypes(include='number').columns.tolist()
categorical_features = X_project.select_dtypes(include='object').columns.tolist()
print('Numerik:', numeric_features)
print('Kategorikal:', categorical_features)

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features)
])
project_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE))
])

In [ ]:
# 2) Pisahkan hold-out test, lalu validasi model hanya di bagian train
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_project, y_project, test_size=0.20, stratify=y_project, random_state=RANDOM_STATE
)
cv_project = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(project_pipeline, X_train_p, y_train_p, cv=cv_project, scoring='average_precision')
print(f'CV PR-AUC: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')

# 3) Fit final di train dan evaluasi satu kali pada hold-out test
project_pipeline.fit(X_train_p, y_train_p)
project_test_proba = project_pipeline.predict_proba(X_test_p)[:, 1]
project_test_pred = project_pipeline.predict(X_test_p)
print(f'Test PR-AUC: {average_precision_score(y_test_p, project_test_proba):.3f}')
print(f'Test balanced accuracy: {balanced_accuracy_score(y_test_p, project_test_pred):.3f}')
print(classification_report(y_test_p, project_test_pred, zero_division=0))

## Checklist sebelum menyimpulkan hasil

- [ ] Definisi target dan horizon waktu sudah jelas.
- [ ] Kolom yang terjadi setelah target atau mengandung jawaban tidak dipakai (hindari *leakage*).
- [ ] Train/test split dilakukan sebelum imputation, encoding, scaling, dan tuning.
- [ ] Metrik sesuai biaya kesalahan bisnis; threshold probabilitas dipilih dari kebutuhan, bukan otomatis 0,5.
- [ ] Model dibandingkan dengan baseline sederhana.
- [ ] Hasil diuji pada data baru/hold-out dan ketidakadilan antar kelompok diperiksa bila relevan.
- [ ] Seed, versi data, fitur, metrik, dan parameter dicatat agar eksperimen dapat diulang.

## Tantangan lanjutan
1. Ganti KNN dengan `RandomForestClassifier` dan bandingkan cross-validation score.
2. Ubah `scoring` pada GridSearchCV menjadi `average_precision`; jelaskan mengapa hasil parameter dapat berubah.
3. Baca CSV sendiri, tentukan `target_col`, lalu gunakan template Extra High. Dokumentasikan alasan menghapus/menyimpan setiap kolom.
4. Untuk data waktu, jangan gunakan split acak; gunakan pemisahan berdasarkan waktu agar masa depan tidak memengaruhi masa lalu.

**Selesai.** Jika Anda sudah mampu menjelaskan tujuan bisnis, sumber data, potensi leakage, metrik, dan limitasi model, Anda sudah melakukan data mining dengan fondasi yang benar.